In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

In [2]:
production_model = joblib.load("../models/catboost_production.pkl")

In [3]:
production = pd.read_csv("../datasets/processed/production_features.csv")

In [4]:
print(production_model.feature_names_)

['State', 'District_Name', 'Year', 'Season', 'Crop', 'Area', 'Avg_Annual_Temperature', 'Avg_Annual_DewPoint', 'Total_Annual_Rainfall', 'Avg_Annual_Wind_Speed', 'Avg_Annual_Surface_Pressure', 'Avg_Grid_Points', 'Log_Area', 'Production_Lag_1', 'Production_Lag_2', 'Production_Lag_3', 'Rolling_Mean_3', 'Rolling_Mean_5', 'Rolling_STD_3', 'Previous_Change', 'Previous_Growth', 'Years_From_Start', 'Year_Squared', 'Temp_Rainfall', 'Temp_DewPoint', 'Rainfall_Wind', 'Area_Rainfall', 'Area_Temperature']


In [5]:
def get_production_history(
    state,
    district,
    crop,
    season
):

    history = (
        production[
            (production["State"] == state) &
            (production["District_Name"] == district) &
            (production["Crop"] == crop) &
            (production["Season"] == season)
        ]
        .sort_values("Year")
        .reset_index(drop=True)
        .copy()
    )

    if history.empty:
        raise ValueError(
            f"No historical data found for {crop} in {district}, {state}"
        )

    return history

In [6]:
production[
    production["State"] == "Andhra Pradesh"
]["District_Name"].unique()

<ArrowStringArray>
[       'Krishna',      'Anantapur',  'East Godavari',   'Vizianagaram',
  'West Godavari',         'Kadapa',        'Kurnool',       'Prakasam',
   'Spsr Nellore',     'Srikakulam', 'Visakhapatanam',       'Chittoor',
         'Guntur']
Length: 13, dtype: str

In [9]:
def prepare_production_features(
    history,
    year,
    area,
    temperature,
    dew_point,
    rainfall,
    wind_speed,
    pressure
):

    last = history.iloc[-1]

    features = {
        "State": last["State"],
        "District_Name": last["District_Name"],
        "Year": year,
        "Season": last["Season"],
        "Crop": last["Crop"],

        # User Input
        "Area": area,

        # Weather
        "Avg_Annual_Temperature": temperature,
        "Avg_Annual_DewPoint": dew_point,
        "Total_Annual_Rainfall": rainfall,
        "Avg_Annual_Wind_Speed": wind_speed,
        "Avg_Annual_Surface_Pressure": pressure,

        "Avg_Grid_Points": last["Avg_Grid_Points"],

        # Engineered Features
        "Log_Area": np.log1p(area),

        "Production_Lag_1": last["Production_Lag_1"],
        "Production_Lag_2": last["Production_Lag_2"],
        "Production_Lag_3": last["Production_Lag_3"],

        "Rolling_Mean_3": last["Rolling_Mean_3"],
        "Rolling_Mean_5": last["Rolling_Mean_5"],
        "Rolling_STD_3": last["Rolling_STD_3"],

        "Previous_Change": last["Previous_Change"],
        "Previous_Growth": last["Previous_Growth"],

        "Years_From_Start": year - history["Year"].min(),

        "Year_Squared": (year - history["Year"].min()) ** 2,

        "Temp_Rainfall": temperature * rainfall,
        "Temp_DewPoint": temperature * dew_point,
        "Rainfall_Wind": rainfall * wind_speed,

        "Area_Rainfall": area * rainfall,
        "Area_Temperature": area * temperature,
    }

    features = pd.DataFrame([features])

    return features[production_model.feature_names_]

In [25]:
production["State"].unique()

<ArrowStringArray>
['Andaman And Nicobar Islands',              'Andhra Pradesh',
           'Arunachal Pradesh',                       'Assam',
                       'Bihar',                  'Chandigarh',
                'Chhattisgarh',      'Dadra And Nagar Haveli',
                         'Goa',                     'Gujarat',
                     'Haryana',            'Himachal Pradesh',
           'Jammu And Kashmir',                   'Jharkhand',
                   'Karnataka',                      'Kerala',
              'Madhya Pradesh',                 'Maharashtra',
                     'Manipur',                   'Meghalaya',
                     'Mizoram',                    'Nagaland',
                      'Odisha',                  'Puducherry',
                      'Punjab',                   'Rajasthan',
                      'Sikkim',                  'Tamil Nadu',
                   'Telangana',                     'Tripura',
               'Uttar Pradesh',     

In [26]:
production[
    production["State"] == "Andhra Pradesh"
]["District_Name"].unique()

<ArrowStringArray>
[       'Krishna',      'Anantapur',  'East Godavari',   'Vizianagaram',
  'West Godavari',         'Kadapa',        'Kurnool',       'Prakasam',
   'Spsr Nellore',     'Srikakulam', 'Visakhapatanam',       'Chittoor',
         'Guntur']
Length: 13, dtype: str

In [27]:
production[
    (production["State"] == "Andhra Pradesh") &
    (production["District_Name"] == "Anantapur")
]["Crop"].unique()

<ArrowStringArray>
[           'Arecanut',           'Arhar/Tur',               'Bajra',
              'Banana',              'Bhindi',             'Brinjal',
             'Cabbage',           'Cashewnut',         'Castor Seed',
        'Citrus Fruit',             'Coconut',           'Coriander',
        'Cotton(Lint)',       'Cowpea(Lobia)',        'Dry Chillies',
                'Gram',              'Grapes',           'Groundnut',
          'Horse-Gram',               'Jowar',               'Korra',
               'Lemon',             'Linseed',               'Maize',
               'Mango',               'Mesta',   'Moong(Green Gram)',
               'Onion',              'Orange',  'Other  Rabi Pulses',
  'Other Fresh Fruits', 'Other Kharif Pulses',      'Other Oilseeds',
              'Papaya',          'Pome Fruit',         'Pome Granet',
              'Potato',                'Ragi',                'Rice',
           'Safflower',               'Samai',              'Sapota',
 

In [28]:
production[
    (production["State"] == "Andhra Pradesh") &
    (production["District_Name"] == "Anantapur") &
    (production["Crop"] == "Groundnut")
]["Season"].unique()

<ArrowStringArray>
['Kharif', 'Rabi']
Length: 2, dtype: str

In [29]:
history = get_production_history(
    state="Andhra Pradesh",
    district="Anantapur",
    crop="Groundnut",
    season="Kharif"
)

history.tail()

,State,District_Name,Year,Season,Crop,Area,Production,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
12,Andhra Pradesh,Anantapur,2010,Kharif,Groundnut,814077.0,446928.0,27.400900,21.119236,45.253963,...,1.077386,2.071646,0.283352,13,169,1239.999315,578.686074,84.456100,615.897904,372.921083
13,Andhra Pradesh,Anantapur,2011,Kharif,Groundnut,733960.0,174682.0,27.246935,19.954323,25.373302,...,0.836824,1.673116,0.221787,14,196,691.344710,543.694142,53.430389,342.697175,368.002858
14,Andhra Pradesh,Anantapur,2012,Kharif,Groundnut,708708.0,295531.0,27.572472,19.865290,28.807112,...,0.856973,1.649587,0.212880,15,225,794.283289,547.735152,59.686098,388.066375,371.434293
15,Andhra Pradesh,Anantapur,2013,Kharif,Groundnut,711145.0,286591.0,27.226873,20.376078,36.435356,...,0.964580,1.801953,0.231027,16,256,992.020811,554.776888,79.193223,490.953052,366.872122
16,Andhra Pradesh,Anantapur,2014,Kharif,Groundnut,550794.0,148714.0,27.607834,20.294256,28.404898,...,0.786140,1.517207,0.199052,17,289,784.197709,560.280451,59.920794,375.487698,364.951214


In [30]:
last = history.iloc[-1]

sample = prepare_production_features(
    history=history,
    year=2016,
    area=100,
    temperature=last["Avg_Annual_Temperature"],
    dew_point=last["Avg_Annual_DewPoint"],
    rainfall=last["Total_Annual_Rainfall"],
    wind_speed=last["Avg_Annual_Wind_Speed"],
    pressure=last["Avg_Annual_Surface_Pressure"]
)

sample

,State,District_Name,Year,Season,Crop,Area,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,Avg_Annual_Wind_Speed,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
0,Andhra Pradesh,Anantapur,2016,Kharif,Groundnut,100,27.607834,20.294256,28.404898,2.109523,...,0.78614,1.517207,0.199052,18,324,784.197709,560.280451,59.920794,2840.4898,2760.7834


In [37]:
def predict_next_production(features):

    predicted_log_production = production_model.predict(features)[0]

    predicted_production = np.expm1(predicted_log_production)

    return predicted_log_production, predicted_production

In [38]:
predicted_log_production, predicted_production = predict_next_production(sample)

print(f"Log Production : {predicted_log_production:.4f}")
print(f"Production     : {predicted_production:.2f}")

Log Production : 5.4651
Production     : 235.30


In [39]:
def update_production_history(
    history,
    predicted_log_production,
    year,
    area,
    temperature,
    dew_point,
    rainfall,
    wind_speed,
    pressure
):

    last = history.iloc[-1].copy()
    new_row = last.copy()

    # ----------------------------------
    # Time
    # ----------------------------------
    new_row["Year"] = year

    # ----------------------------------
    # User Input
    # ----------------------------------
    new_row["Area"] = area
    new_row["Log_Area"] = np.log1p(area)

    # ----------------------------------
    # Predicted Production
    # ----------------------------------
    new_row["Log_Production"] = predicted_log_production
    new_row["Production"] = np.expm1(predicted_log_production)

    # ----------------------------------
    # Lag Features
    # ----------------------------------
    new_row["Production_Lag_3"] = last["Production_Lag_2"]
    new_row["Production_Lag_2"] = last["Production_Lag_1"]
    new_row["Production_Lag_1"] = predicted_log_production

    # ----------------------------------
    # Weather
    # ----------------------------------
    new_row["Avg_Annual_Temperature"] = temperature
    new_row["Avg_Annual_DewPoint"] = dew_point
    new_row["Total_Annual_Rainfall"] = rainfall
    new_row["Avg_Annual_Wind_Speed"] = wind_speed
    new_row["Avg_Annual_Surface_Pressure"] = pressure

    # ----------------------------------
    # Time Features
    # ----------------------------------
    start_year = history["Year"].min()

    new_row["Years_From_Start"] = year - start_year
    new_row["Year_Squared"] = (year - start_year) ** 2

    # ----------------------------------
    # Interaction Features
    # ----------------------------------
    new_row["Temp_Rainfall"] = temperature * rainfall
    new_row["Temp_DewPoint"] = temperature * dew_point
    new_row["Rainfall_Wind"] = rainfall * wind_speed

    new_row["Area_Rainfall"] = area * rainfall
    new_row["Area_Temperature"] = area * temperature

    # ----------------------------------
    # Append New Row
    # ----------------------------------
    history = pd.concat(
        [history, pd.DataFrame([new_row])],
        ignore_index=True
    )

    # ----------------------------------
    # Rolling Features
    # ----------------------------------
    log_productions = history["Log_Production"]

    history.loc[
        history.index[-1],
        "Rolling_Mean_3"
    ] = log_productions.iloc[-3:].mean()

    history.loc[
        history.index[-1],
        "Rolling_Mean_5"
    ] = log_productions.iloc[-5:].mean()

    history.loc[
        history.index[-1],
        "Rolling_STD_3"
    ] = log_productions.iloc[-3:].std()

    # ----------------------------------
    # Previous Change
    # ----------------------------------
    previous_change = (
        predicted_log_production -
        last["Log_Production"]
    )

    history.loc[
        history.index[-1],
        "Previous_Change"
    ] = previous_change

    # ----------------------------------
    # Previous Growth
    # ----------------------------------
    if last["Log_Production"] != 0:
        history.loc[
            history.index[-1],
            "Previous_Growth"
        ] = (
            previous_change /
            last["Log_Production"]
        )
    else:
        history.loc[
            history.index[-1],
            "Previous_Growth"
        ] = 0

    return history

In [40]:
def forecast_production(
    state,
    district,
    crop,
    season,
    area,
    start_year,
    years,
    weather_data
):

    if len(weather_data) < years:
        raise ValueError(
            f"Weather data contains {len(weather_data)} rows but {years} forecast years were requested."
        )

    history = get_production_history(
        state,
        district,
        crop,
        season
    )

    forecasts = []

    year = start_year

    for i in range(years):

        weather = weather_data.iloc[i]

        features = prepare_production_features(
            history=history,
            year=year,
            area=area,
            temperature=weather["Avg_Annual_Temperature"],
            dew_point=weather["Avg_Annual_DewPoint"],
            rainfall=weather["Total_Annual_Rainfall"],
            wind_speed=weather["Avg_Annual_Wind_Speed"],
            pressure=weather["Avg_Annual_Surface_Pressure"]
        )

        predicted_log_production, predicted_production = predict_next_production(
            features
        )

        forecasts.append({

            "Forecast_Year": i + 1,

            "State": state,

            "District": district,

            "Crop": crop,

            "Season": season,

            "Year": year,

            "Area": area,

            "Predicted_Log_Production": predicted_log_production,

            "Predicted_Production": predicted_production

        })

        history = update_production_history(
            history=history,
            predicted_log_production=predicted_log_production,
            year=year,
            area=area,
            temperature=weather["Avg_Annual_Temperature"],
            dew_point=weather["Avg_Annual_DewPoint"],
            rainfall=weather["Total_Annual_Rainfall"],
            wind_speed=weather["Avg_Annual_Wind_Speed"],
            pressure=weather["Avg_Annual_Surface_Pressure"]
        )

        year += 1

    return pd.DataFrame(forecasts), history

In [41]:
last = history.iloc[-1]

weather_data = pd.DataFrame([
    {
        "Avg_Annual_Temperature": last["Avg_Annual_Temperature"],
        "Avg_Annual_DewPoint": last["Avg_Annual_DewPoint"],
        "Total_Annual_Rainfall": last["Total_Annual_Rainfall"],
        "Avg_Annual_Wind_Speed": last["Avg_Annual_Wind_Speed"],
        "Avg_Annual_Surface_Pressure": last["Avg_Annual_Surface_Pressure"]
    }
] * 5)

In [42]:
forecast, updated_history = forecast_production(
    state="Andhra Pradesh",
    district="Anantapur",      # Replace with your valid district
    crop="Groundnut",          # Replace with your valid crop
    season="Kharif",
    area=100,
    start_year=2016,
    years=5,
    weather_data=weather_data
)

forecast

,Forecast_Year,State,District,Crop,Season,Year,Area,Predicted_Log_Production,Predicted_Production
0,1,Andhra Pradesh,Anantapur,Groundnut,Kharif,2016,100,5.465101,235.299629
1,2,Andhra Pradesh,Anantapur,Groundnut,Kharif,2017,100,5.596369,268.446330
2,3,Andhra Pradesh,Anantapur,Groundnut,Kharif,2018,100,5.552129,256.785891
3,4,Andhra Pradesh,Anantapur,Groundnut,Kharif,2019,100,5.546865,255.432313
4,5,Andhra Pradesh,Anantapur,Groundnut,Kharif,2020,100,5.449297,231.594505
